# Phase 5 — PPO Testing and Crop Recommendation

This Google Colab notebook loads the trained PPO model from Phase 4 and produces crop recommendations **without retraining**.

### Upload when requested
1. `proceed_data.csv` — the same processed dataset used in Phase 4.
2. `Farmer_RL_Phase4_Complete.zip` — downloaded at the end of Phase 4.

The notebook will:
- restore the PPO model and `VecNormalize` statistics;
- verify dataset and crop compatibility;
- evaluate the trained agent;
- recommend a crop for farmer conditions;
- test multiple scenarios;
- export recommendation CSV files.

In [ ]:
# Install maintained RL libraries in Google Colab.
!pip -q install "gymnasium>=1.0,<2.0" "stable-baselines3>=2.4,<3.0" pandas numpy matplotlib

In [ ]:
import json
import shutil
import warnings
import zipfile
from pathlib import Path

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from gymnasium import spaces

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

Path("phase4_files").mkdir(exist_ok=True)
Path("phase5_results").mkdir(exist_ok=True)

print("Gymnasium:", gym.__version__)

## 1. Upload the processed dataset

In [ ]:
from google.colab import files

print("Upload proceed_data.csv (the same CSV used during Phase 4).")
uploaded = files.upload()

csv_names = [name for name in uploaded if name.lower().endswith(".csv")]
if not csv_names:
    raise RuntimeError("No CSV file uploaded. Please rerun this cell and upload proceed_data.csv.")

CSV_PATH = csv_names[0]
print("Dataset selected:", CSV_PATH)

In [ ]:
REQUIRED_COLUMNS = ["Crop", "Annual_Rainfall", "Yield", "Modal_Price", "Profit"]
OPTIONAL_NUMERIC_COLUMNS = ["Area", "Production", "Fertilizer", "Pesticide", "Revenue", "Cost"]

df = pd.read_csv(CSV_PATH)
df.columns = df.columns.str.strip()

missing = [column for column in REQUIRED_COLUMNS if column not in df.columns]
if missing:
    raise ValueError(f"Dataset is missing required columns: {missing}")

for column in REQUIRED_COLUMNS[1:] + [c for c in OPTIONAL_NUMERIC_COLUMNS if c in df.columns]:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df["Crop"] = df["Crop"].astype(str).str.strip()
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=REQUIRED_COLUMNS).reset_index(drop=True)

if df.empty:
    raise ValueError("No valid rows remain after data cleaning.")
if df["Crop"].nunique() < 2:
    raise ValueError("At least two different crops are required.")

print("Rows:", len(df))
print("Crops:", df["Crop"].nunique())
display(df.head())

## 2. Upload Phase 4 trained outputs

In [ ]:
from google.colab import files

print("Upload Farmer_RL_Phase4_Complete.zip downloaded from Phase 4.")
uploaded_model = files.upload()

zip_names = [name for name in uploaded_model if name.lower().endswith(".zip")]
if not zip_names:
    raise RuntimeError("No ZIP file uploaded. Upload Farmer_RL_Phase4_Complete.zip.")

PHASE4_ZIP = zip_names[0]

# Clear previous extraction and extract the uploaded package.
extract_dir = Path("phase4_files")
if extract_dir.exists():
    shutil.rmtree(extract_dir)
extract_dir.mkdir()

with zipfile.ZipFile(PHASE4_ZIP, "r") as archive:
    archive.extractall(extract_dir)

print("Extracted:", PHASE4_ZIP)

In [ ]:
# Locate files even if the ZIP contains nested folders.
def find_first(patterns):
    for pattern in patterns:
        matches = list(Path("phase4_files").rglob(pattern))
        if matches:
            return matches[0]
    return None

BEST_MODEL_PATH = find_first(["best_model.zip"])
FINAL_MODEL_PATH = find_first(["ppo_farmer_model.zip"])
VEC_NORMALIZE_PATH = find_first(["vec_normalize.pkl"])
CONFIG_PATH = find_first(["training_config.json"])

MODEL_PATH = BEST_MODEL_PATH or FINAL_MODEL_PATH

if MODEL_PATH is None:
    raise FileNotFoundError("Could not find best_model.zip or ppo_farmer_model.zip inside the Phase 4 package.")
if VEC_NORMALIZE_PATH is None:
    raise FileNotFoundError("Could not find vec_normalize.pkl inside the Phase 4 package.")

print("PPO model:", MODEL_PATH)
print("Normalization:", VEC_NORMALIZE_PATH)
print("Configuration:", CONFIG_PATH if CONFIG_PATH else "Not found; defaults will be used")

training_config = {}
if CONFIG_PATH:
    training_config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
    display(training_config)

## 3. Define the same FarmEnv used during Phase 4

In [ ]:
class FarmEnv(gym.Env):
    """Crop-selection environment used in Phase 4."""

    metadata = {"render_modes": ["human"]}

    def __init__(
        self,
        data: pd.DataFrame,
        horizon_years: int = 20,
        initial_savings: float = 100_000.0,
        annual_living_cost: float | None = None,
        rainfall_volatility: float = 0.12,
        price_volatility: float = 0.10,
        rotation_penalty_rate: float = 0.20,
        reward_scale: float | None = None,
        render_mode: str | None = None,
    ):
        super().__init__()
        self.data = data.copy().reset_index(drop=True)
        required = {"Crop", "Annual_Rainfall", "Yield", "Modal_Price", "Profit"}
        missing = required.difference(self.data.columns)
        if missing:
            raise ValueError(f"FarmEnv missing columns: {sorted(missing)}")

        self.crops = sorted(self.data["Crop"].astype(str).unique().tolist())
        if len(self.crops) < 2:
            raise ValueError("FarmEnv requires at least two crops.")

        self.crop_to_index = {crop: i for i, crop in enumerate(self.crops)}
        self.crop_rows = {
            crop: self.data.index[self.data["Crop"] == crop].to_numpy()
            for crop in self.crops
        }
        self.horizon_years = int(horizon_years)
        self.initial_savings = float(initial_savings)
        self.rainfall_volatility = float(rainfall_volatility)
        self.price_volatility = float(price_volatility)
        self.rotation_penalty_rate = float(rotation_penalty_rate)
        self.render_mode = render_mode

        typical_profit = float(np.nanmedian(np.abs(self.data["Profit"].to_numpy())))
        self.profit_scale = max(typical_profit, 1.0)
        self.reward_scale = float(reward_scale or self.profit_scale)
        self.annual_living_cost = float(
            annual_living_cost if annual_living_cost is not None else 0.15 * self.profit_scale
        )
        self.savings_scale = max(
            abs(self.initial_savings) + self.horizon_years * self.profit_scale,
            1.0,
        )

        self.rain_min, self.rain_max = self._safe_bounds("Annual_Rainfall")
        self.price_min, self.price_max = self._safe_bounds("Modal_Price")
        self.yield_min, self.yield_max = self._safe_bounds("Yield")

        self.action_space = spaces.Discrete(len(self.crops))
        self.observation_space = spaces.Box(
            low=np.full(6, -1.0, dtype=np.float32),
            high=np.full(6, 1.0, dtype=np.float32),
            dtype=np.float32,
        )
        self.current_year = 0
        self.savings = self.initial_savings
        self.last_crop_index = -1
        self.current_context = None
        self.history = []

    def _safe_bounds(self, column: str):
        values = self.data[column].astype(float).to_numpy()
        low, high = float(np.nanmin(values)), float(np.nanmax(values))
        if not np.isfinite(low) or not np.isfinite(high):
            return 0.0, 1.0
        if np.isclose(low, high):
            high = low + 1.0
        return low, high

    @staticmethod
    def _scale(value: float, low: float, high: float) -> float:
        scaled = 2.0 * (float(value) - low) / (high - low) - 1.0
        return float(np.clip(scaled, -1.0, 1.0))

    def _sample_context(self):
        row = self.data.iloc[int(self.np_random.integers(0, len(self.data)))]
        return {
            "rainfall": float(row["Annual_Rainfall"]),
            "price": float(row["Modal_Price"]),
            "yield": float(row["Yield"]),
        }

    def _get_observation(self):
        context = self.current_context
        progress = 2.0 * (self.current_year / max(self.horizon_years, 1)) - 1.0
        savings_scaled = np.clip(self.savings / self.savings_scale, -1.0, 1.0)
        last_crop_scaled = (
            -1.0 if self.last_crop_index < 0
            else self._scale(self.last_crop_index, 0, max(len(self.crops) - 1, 1))
        )
        return np.array([
            progress,
            savings_scaled,
            self._scale(context["rainfall"], self.rain_min, self.rain_max),
            self._scale(context["price"], self.price_min, self.price_max),
            self._scale(context["yield"], self.yield_min, self.yield_max),
            last_crop_scaled,
        ], dtype=np.float32)

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self.current_year = 0
        self.savings = self.initial_savings
        self.last_crop_index = -1
        self.current_context = self._sample_context()
        self.history = []
        return self._get_observation(), {
            "year": self.current_year,
            "savings": self.savings,
            "crops": self.crops,
        }

    def step(self, action):
        action = int(action)
        if not self.action_space.contains(action):
            raise ValueError(f"Invalid action {action}.")

        crop = self.crops[action]
        row = self.data.iloc[int(self.np_random.choice(self.crop_rows[crop]))]
        base_profit = float(row["Profit"])
        crop_rainfall = max(float(row["Annual_Rainfall"]), 1e-6)
        crop_price = max(float(row["Modal_Price"]), 1e-6)

        rain_noise = float(self.np_random.normal(1.0, self.rainfall_volatility))
        price_noise = float(self.np_random.normal(1.0, self.price_volatility))
        rainfall_ratio = (self.current_context["rainfall"] * rain_noise) / crop_rainfall
        price_ratio = (self.current_context["price"] * price_noise) / crop_price
        climate_factor = float(np.clip(0.5 + 0.5 * rainfall_ratio, 0.35, 1.35))
        market_factor = float(np.clip(price_ratio, 0.50, 1.50))
        realised_profit = base_profit * climate_factor * market_factor

        rotation_penalty = 0.0
        if action == self.last_crop_index:
            rotation_penalty = abs(realised_profit) * self.rotation_penalty_rate

        net_income = realised_profit - rotation_penalty - self.annual_living_cost
        self.savings += net_income
        self.current_year += 1

        terminated = self.savings <= 0.0
        bankruptcy_penalty = self.profit_scale if terminated else 0.0
        truncated = self.current_year >= self.horizon_years
        reward = (net_income - bankruptcy_penalty) / self.reward_scale

        record = {
            "year": self.current_year,
            "crop": crop,
            "base_profit": base_profit,
            "realised_profit": realised_profit,
            "rotation_penalty": rotation_penalty,
            "living_cost": self.annual_living_cost,
            "net_income": net_income,
            "savings": self.savings,
            "reward": reward,
        }
        self.history.append(record)
        self.last_crop_index = action
        if not (terminated or truncated):
            self.current_context = self._sample_context()

        return self._get_observation(), float(reward), bool(terminated), bool(truncated), record.copy()

    def get_history(self):
        return pd.DataFrame(self.history)

## 4. Restore PPO and normalization statistics

In [ ]:
HORIZON_YEARS = int(training_config.get("horizon_years", 20))
INITIAL_SAVINGS = float(training_config.get("initial_savings", 100_000.0))


def make_env(seed_offset=0):
    def _init():
        env = FarmEnv(
            df,
            horizon_years=HORIZON_YEARS,
            initial_savings=INITIAL_SAVINGS,
        )
        env = Monitor(env)
        env.reset(seed=SEED + seed_offset)
        return env
    return _init

raw_vec_env = DummyVecEnv([make_env(3000)])
norm_env = VecNormalize.load(str(VEC_NORMALIZE_PATH), raw_vec_env)
norm_env.training = False
norm_env.norm_reward = False

model = PPO.load(str(MODEL_PATH), env=norm_env)
base_env = norm_env.venv.envs[0].unwrapped

trained_crops = training_config.get("crops")
if trained_crops is not None and list(trained_crops) != base_env.crops:
    raise ValueError(
        "Crop order differs from Phase 4. Use exactly the same proceed_data.csv used for training.
"
        f"Training crops: {trained_crops}
Current crops: {base_env.crops}"
    )

if model.action_space.n != len(base_env.crops):
    raise ValueError(
        f"Model expects {model.action_space.n} actions, but the dataset has {len(base_env.crops)} crops."
    )

print("✅ PPO model restored successfully.")
print("Model actions:", model.action_space.n)
print("Crop order:")
for index, crop in enumerate(base_env.crops):
    print(f"  {index}: {crop}")

## 5. Test the trained PPO agent for one complete episode

In [ ]:
obs = norm_env.reset()
done = np.array([False])
episode_records = []
total_reward = 0.0

while not bool(done[0]):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, infos = norm_env.step(action)
    total_reward += float(reward[0])

    info = infos[0]
    if "crop" in info:
        episode_records.append({
            "year": int(info["year"]),
            "crop": info["crop"],
            "realised_profit": float(info["realised_profit"]),
            "rotation_penalty": float(info["rotation_penalty"]),
            "net_income": float(info["net_income"]),
            "savings": float(info["savings"]),
            "reward": float(reward[0]),
        })

test_history = pd.DataFrame(episode_records)
test_history.to_csv("phase5_results/ppo_test_episode.csv", index=False)

display(test_history)
print("Total episode reward:", round(total_reward, 4))
if not test_history.empty:
    print("Final savings:", round(float(test_history["savings"].iloc[-1]), 2))

## 6. Crop recommendation function

In [ ]:
def _clip_to_dataset(value, column):
    low = float(df[column].min())
    high = float(df[column].max())
    return float(np.clip(float(value), low, high))


def recommend_crop(
    rainfall,
    market_price,
    expected_yield,
    current_savings=INITIAL_SAVINGS,
    current_year=0,
    previous_crop=None,
):
    """Recommend one crop using the trained PPO policy."""

    rainfall = _clip_to_dataset(rainfall, "Annual_Rainfall")
    market_price = _clip_to_dataset(market_price, "Modal_Price")
    expected_yield = _clip_to_dataset(expected_yield, "Yield")
    current_year = int(np.clip(current_year, 0, HORIZON_YEARS - 1))

    # Reset, then replace the randomly sampled state with farmer inputs.
    norm_env.reset()
    env = norm_env.venv.envs[0].unwrapped
    env.current_year = current_year
    env.savings = float(current_savings)
    env.current_context = {
        "rainfall": rainfall,
        "price": market_price,
        "yield": expected_yield,
    }

    if previous_crop is None or str(previous_crop).strip() == "":
        env.last_crop_index = -1
        previous_crop_name = "None"
    else:
        previous_crop = str(previous_crop).strip()
        if previous_crop not in env.crop_to_index:
            raise ValueError(f"Unknown previous crop. Choose from: {env.crops}")
        env.last_crop_index = env.crop_to_index[previous_crop]
        previous_crop_name = previous_crop

    raw_observation = env._get_observation().reshape(1, -1)
    normalized_observation = norm_env.normalize_obs(raw_observation.copy())
    action, _ = model.predict(normalized_observation, deterministic=True)
    action_index = int(np.asarray(action).reshape(-1)[0])
    crop = env.crops[action_index]

    # Explain the recommendation with a deterministic estimate based on crop medians.
    crop_data = df[df["Crop"] == crop]
    base_profit = float(crop_data["Profit"].median())
    crop_rainfall = max(float(crop_data["Annual_Rainfall"].median()), 1e-6)
    crop_price = max(float(crop_data["Modal_Price"].median()), 1e-6)

    rainfall_ratio = rainfall / crop_rainfall
    price_ratio = market_price / crop_price
    climate_factor = float(np.clip(0.5 + 0.5 * rainfall_ratio, 0.35, 1.35))
    market_factor = float(np.clip(price_ratio, 0.50, 1.50))
    estimated_profit = base_profit * climate_factor * market_factor

    rotation_penalty = 0.0
    if previous_crop_name == crop:
        rotation_penalty = abs(estimated_profit) * env.rotation_penalty_rate

    estimated_net_income = estimated_profit - rotation_penalty - env.annual_living_cost
    estimated_savings = float(current_savings) + estimated_net_income

    return {
        "recommended_crop": crop,
        "action_index": action_index,
        "rainfall_used": rainfall,
        "market_price_used": market_price,
        "yield_used": expected_yield,
        "current_savings": float(current_savings),
        "current_year": current_year,
        "previous_crop": previous_crop_name,
        "estimated_base_profit": base_profit,
        "estimated_realised_profit": estimated_profit,
        "estimated_rotation_penalty": rotation_penalty,
        "annual_living_cost": env.annual_living_cost,
        "estimated_net_income": estimated_net_income,
        "estimated_savings_after_year": estimated_savings,
    }

## 7. Enter farmer details and get a recommendation

In [ ]:
# Change these values to test a farmer's conditions.
FARMER_RAINFALL = float(df["Annual_Rainfall"].median())
FARMER_MARKET_PRICE = float(df["Modal_Price"].median())
FARMER_EXPECTED_YIELD = float(df["Yield"].median())
FARMER_CURRENT_SAVINGS = INITIAL_SAVINGS
FARMER_CURRENT_YEAR = 0
FARMER_PREVIOUS_CROP = None  # Example: "Rice"; use None for the first year.

recommendation = recommend_crop(
    rainfall=FARMER_RAINFALL,
    market_price=FARMER_MARKET_PRICE,
    expected_yield=FARMER_EXPECTED_YIELD,
    current_savings=FARMER_CURRENT_SAVINGS,
    current_year=FARMER_CURRENT_YEAR,
    previous_crop=FARMER_PREVIOUS_CROP,
)

recommendation_df = pd.DataFrame([recommendation])
recommendation_df.to_csv("phase5_results/single_crop_recommendation.csv", index=False)
display(recommendation_df)

print("
🌾 Recommended crop:", recommendation["recommended_crop"])
print("Estimated net income:", round(recommendation["estimated_net_income"], 2))
print("Estimated savings after year:", round(recommendation["estimated_savings_after_year"], 2))
print("
Note: financial values are model estimates based on historical medians, not guaranteed outcomes.")

## 8. Test low, normal, and high-condition scenarios

In [ ]:
scenario_inputs = [
    {
        "scenario": "Low conditions",
        "rainfall": float(df["Annual_Rainfall"].quantile(0.25)),
        "market_price": float(df["Modal_Price"].quantile(0.25)),
        "expected_yield": float(df["Yield"].quantile(0.25)),
    },
    {
        "scenario": "Normal conditions",
        "rainfall": float(df["Annual_Rainfall"].median()),
        "market_price": float(df["Modal_Price"].median()),
        "expected_yield": float(df["Yield"].median()),
    },
    {
        "scenario": "High conditions",
        "rainfall": float(df["Annual_Rainfall"].quantile(0.75)),
        "market_price": float(df["Modal_Price"].quantile(0.75)),
        "expected_yield": float(df["Yield"].quantile(0.75)),
    },
]

scenario_results = []
for scenario in scenario_inputs:
    result = recommend_crop(
        rainfall=scenario["rainfall"],
        market_price=scenario["market_price"],
        expected_yield=scenario["expected_yield"],
        current_savings=INITIAL_SAVINGS,
        current_year=0,
        previous_crop=None,
    )
    result["scenario"] = scenario["scenario"]
    scenario_results.append(result)

scenario_df = pd.DataFrame(scenario_results)
scenario_df = scenario_df[[
    "scenario", "recommended_crop", "rainfall_used", "market_price_used",
    "yield_used", "estimated_realised_profit", "estimated_net_income",
    "estimated_savings_after_year"
]]
scenario_df.to_csv("phase5_results/scenario_recommendations.csv", index=False)
display(scenario_df)

## 9. Visualize scenario recommendations

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(scenario_df["scenario"], scenario_df["estimated_net_income"])
plt.ylabel("Estimated net income")
plt.title("PPO Recommendation Estimates by Scenario")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("phase5_results/scenario_net_income.png", dpi=160)
plt.show()

crop_frequency = test_history["crop"].value_counts() if not test_history.empty else pd.Series(dtype=int)
if not crop_frequency.empty:
    plt.figure(figsize=(10, 5))
    crop_frequency.plot(kind="bar")
    plt.xlabel("Crop")
    plt.ylabel("Selections")
    plt.title("Crop Selections in PPO Test Episode")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig("phase5_results/test_crop_frequency.png", dpi=160)
    plt.show()

## 10. Package and download Phase 5 results

In [ ]:
from google.colab import files

with zipfile.ZipFile("Farmer_RL_Phase5_Results.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for path in Path("phase5_results").rglob("*"):
        if path.is_file():
            archive.write(path, path.as_posix())

print("✅ Created Farmer_RL_Phase5_Results.zip")
files.download("Farmer_RL_Phase5_Results.zip")

## Phase 5 complete

Keep these files for the next phase:
- `Farmer_RL_Phase4_Complete.zip`
- `Farmer_RL_Phase5_Results.zip`
- `proceed_data.csv`

The next phase is **Phase 6 — detailed performance evaluation and PPO comparison**.